# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwasay45/flyrankinternship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question**  
Which pages should an editor review first for content refresh, given observable search and engagement signals?

**Decision supported**  
Prioritizing limited editor capacity — which pages get looked at this week.

**Unit of analysis**  
One page (content item).

**Output**  
A ranked review queue: priority score + primary reason code + suggested action  
(`refresh` | `refresh_and_review_ctr` | `expand_and_refresh` | `monitor`).

**Cost of a wrong call**  
- False positive → wasted editor hours.  
- False negative → continued traffic loss that might have been recoverable.

**Why data / ML helps**  
A single hand rule (e.g. “stale × visible”) captures only one pattern. Real pages combine age, volume, position, and CTR in ways that are too messy for a short if-statement. A readable ranking model can combine those signals and still stay explainable.

In [8]:
# Section 1 is framing only — no code required.
print("Section 1: Question framing complete.")

Section 1: Question framing complete.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Primary dataset**  
Starter release: `data/raw/content_refresh_anonymized.csv`  
~30,000 pages × 44 columns, 32 pseudonymized clients, trailing-90-day metrics.

**Warehouse context**  
FlyRank internship warehouse on Hugging Face (~79M daily rows).  
Development used mid-panel months (e.g. 2026-03); final month kept sealed.

**What was deliberately excluded**
- Product health / priority / action flags (not in release; would leak past decisions)
- `trend_pct` and `trend_direction` as features (they define the label)
- Raw client names, domains, URLs, titles, private queries
- IDs as model features (grouping and client-holdout only)

All identifiers are pseudonymous. No client-identifying information appears in this work.

In [9]:
import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/abdulwasay45/flyrankinternship.git"
    REPO_DIR = "flyrankinternship"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")
print(f"Clients: {df['client_id'].nunique()}")
df[["content_id", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]].head(3)

Rows: 30,000 | Columns: 45
Declining rate: 54.2%
Clients: 32


,content_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,content_304f48230142,3803,20,10.6,0.76,down
1,content_a1fb4e703a9e,15320,25,20.3,0.05,down
2,content_9aa793d4d895,12581,20,36.5,0.09,down


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label / proxy**  
`is_declining_label` = 1 when `trend_direction == "down"`  
(observed outcome from last-30 vs prev-30 impressions). Not a product rule.

**Features (safe, pre-decision only)**  
Volume (impressions, clicks, sessions), age / freshness, average position & tier, CTR, engagement/scroll, content-type / intent.  
Heavy-tailed volume columns use log1p. No future-window or label-derived inputs.

**Baseline**  
Transparent rule:  
`score = stale(≥180d) × visible(≥500 imp) × log(impressions)`  
+ boost for low-CTR visible pages.  
Each row gets one primary reason code and an action label. This is the bar every model must beat.

**Models**  
Logistic Regression · Decision Tree (max_depth=5) · Random Forest (max_depth=10).  
Complexity added only when it improves the ranking metric.

**Validation**  
Client-holdout (~20% of clients held out entirely).  
Also compared to row-holdout to show the generalization gap.

**Leakage checks**  
- `trend_pct` / `trend_direction` never in the feature matrix  
- Deliberate-leak test: adding a label-derived column jumps Precision@50; removing it restores the honest number  
- No product flags as features

**Primary metric**  
Precision@50 (and Precision@20), always reported next to the base rate.

In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# Safe feature matrix
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    if col in X_num.columns:
        X_num[f"log_{col}"] = np.log1p(X_num[col])
X_cat = pd.get_dummies(df[categorical_features].fillna("unknown").astype(str), dtype=float)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

# Leakage guard
forbidden = {"trend_pct", "trend_direction", "is_declining_label"}
print("Forbidden columns in features:", set(X.columns) & forbidden or "none (good)")

# Baseline score
def baseline_score(frame):
    stale   = (frame["days_since_last_update"] >= 180).astype(int)
    visible = (frame["impressions_90d"] >= 500).astype(int)
    low_ctr = ((frame["impressions_90d"] >= 500) & (frame["avg_position"] > 0) &
               (frame["avg_position"] <= 20) & (frame["ctr"] < 0.5)).astype(int)
    return stale * visible * np.log1p(frame["impressions_90d"]) + low_ctr * np.log1p(frame["impressions_90d"]) * 0.5

df["baseline_score"] = baseline_score(df)
print("Baseline score computed. Methodology section ready.")

Forbidden columns in features: none (good)
Baseline score computed. Methodology section ready.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Comparison on the same client-holdout test set**

All metrics below use the identical held-out clients.  
Base rate is reported next to every Precision@K number.

(After you run the code cell, the live table appears under it.  
Those numbers are the authoritative result for the paper.)

In [11]:
RANDOM_STATE = 42

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(y_true)[order[:min(k, len(y_true))]]
    return float(topk.mean()) if len(topk) else 0.0

# Client-holdout split
client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])
test_mask = client_series.isin(test_clients).to_numpy()
tr_idx = np.where(~test_mask)[0]
te_idx = np.where(test_mask)[0]

if len(tr_idx) == 0 or len(te_idx) == 0 or y.iloc[tr_idx].nunique() < 2 or y.iloc[te_idx].nunique() < 2:
    tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    split_name = "stratified_row_holdout"
else:
    split_name = "client_holdout"

X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
base_te = df.iloc[te_idx]["baseline_score"].to_numpy()

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=150, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

rows = [{
    "model": "baseline_rule",
    "precision_at_20": precision_at_k(y_te, base_te, 20),
    "precision_at_50": precision_at_k(y_te, base_te, 50),
    "roc_auc": float("nan"),
    "test_base_rate": float(y_te.mean()),
}]
for name, model in models.items():
    model.fit(X_tr, y_tr)
    p = model.predict_proba(X_te)[:, 1]
    rows.append({
        "model": name,
        "precision_at_20": precision_at_k(y_te, p, 20),
        "precision_at_50": precision_at_k(y_te, p, 50),
        "roc_auc": float(roc_auc_score(y_te, p)),
        "test_base_rate": float(y_te.mean()),
    })

results = pd.DataFrame(rows).sort_values("precision_at_50", ascending=False)
print(f"Split: {split_name} | test n = {len(y_te):,}")
print(results.round(3).to_string(index=False))
print(f"\nBest by Precision@50: {results.iloc[0]['model']}")

Split: client_holdout | test n = 2,325
              model  precision_at_20  precision_at_50  roc_auc  test_base_rate
      random_forest             0.90             0.72    0.749           0.391
logistic_regression             0.35             0.32    0.700           0.391
      baseline_rule             0.20             0.30      NaN           0.391

Best by Precision@50: random_forest


## 5. Limitations

*What this work cannot claim.*

**What this work cannot claim**
- It does not prove that a refresh *causes* recovery (no causal / experimental design).
- It does not predict Google’s ranking algorithm.
- Starter label and features share a trailing window; a production design would enforce strict past-feature → future-label windows on the full warehouse.
- Client histories are unbalanced; short-history clients are noisier.
- Low-volume pages make CTR and trend estimates unstable — volume floors are required.

**Safe language used throughout**  
observed · measured · directional · decision-support

In [12]:
print("Limitations stated. No causal claims.")

Limitations stated. No causal claims.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**Action playbook (how an editor uses the output)**

| Action | Typical reason code | What the editor does |
|--------|---------------------|----------------------|
| `refresh` | stale_visible_page / declining_with_demand | Open page; update outdated sections if warranted |
| `refresh_and_review_ctr` | low_ctr_visible_page | Check title / meta / intent before a full rewrite |
| `expand_and_refresh` | thin_visible_page | Add missing depth on a page that already has demand |
| `monitor` | general_refresh_review | No urgent action this cycle |

**Human review is mandatory before any publish.**  
The model never auto-publishes, deletes, or noindexes.

**What should NOT be automated**  
Rewriting and publishing content · bulk title changes without a test plan · treating the score as an author performance review · acting on ultra-low-volume pages where noise dominates.

**Monitoring / retrain triggers**  
Precision@50 falling below the frozen baseline for two cycles · sharp shifts in reason-code mix · new content types or clients never seen in training · large feature distribution drift.

In [13]:
# Build a short top-10 preview of the playbook queue
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
low_ctr = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0) &
           (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).astype(int)

df["playbook_score"] = (
    stale * visible * np.log1p(df["impressions_90d"])
    + low_ctr * np.log1p(df["impressions_90d"]) * 0.5
)

def reason(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return "low_ctr_visible_page"
    return "general_refresh_review"

df["reason_code"] = df.apply(reason, axis=1)
df["suggested_action"] = df["reason_code"].map({
    "stale_visible_page": "refresh",
    "low_ctr_visible_page": "refresh_and_review_ctr",
    "general_refresh_review": "monitor",
})
df["playbook_rank"] = df["playbook_score"].rank(method="first", ascending=False).astype(int)

top10 = df.sort_values("playbook_rank").head(10)
print("Top-10 ranked recommendations (preview):")
print(top10[["playbook_rank", "suggested_action", "reason_code",
             "impressions_90d", "days_since_last_update", "ctr"]].to_string(index=False))

Top-10 ranked recommendations (preview):
 playbook_rank suggested_action        reason_code  impressions_90d  days_since_last_update  ctr
             1          refresh stale_visible_page            61678                     194 0.15
             2          refresh stale_visible_page            13299                     193 0.49
             3          refresh stale_visible_page             7558                     193 0.20
             4          refresh stale_visible_page             4556                     194 0.33
             5          refresh stale_visible_page             1697                     193 0.12
             6          refresh stale_visible_page            59472                     194 0.13
             7          refresh stale_visible_page             1408                     183 0.28
             8          refresh stale_visible_page              954                     301 0.42
             9          refresh stale_visible_page            25715                   

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

**Artifacts generated for the deployed paper**
- `work/outputs/w05_model_metrics.json` — model vs baseline metrics
- `work/outputs/w06_split_comparison.json` — row-holdout vs client-holdout
- `work/outputs/w07_playbook_metrics.json` — action / reason distributions
- `work/outputs/action_playbook_queue.csv` — full ranked queue (regenerated; kept out of git by design)
- `work/figures/action_distribution.png` — optional chart for the paper
- Deployed paper: see `submission/paper_url.txt`
- Repo: https://github.com/abdulwasay45/flyrankinternship

**Acknowledgments**  
Built on the FlyRank ML Internship dataset — https://flyrank.ai

In [14]:
import json, os
os.makedirs("work/outputs", exist_ok=True)

# Quick receipt so the paper numbers have a home
receipt = {
    "split": split_name if "split_name" in dir() else "client_holdout",
    "results_table": results.to_dict(orient="records") if "results" in dir() else [],
    "top10_preview": top10[["playbook_rank", "suggested_action", "reason_code"]].to_dict(orient="records") if "top10" in dir() else [],
}
with open("work/outputs/capstone_receipt.json", "w") as f:
    json.dump(receipt, f, indent=2)
print("Wrote work/outputs/capstone_receipt.json")
print("Artifacts section complete.")

Wrote work/outputs/capstone_receipt.json
Artifacts section complete.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
